# Netflix Data Quality Assessment
## Comprehensive Analysis of Data Quality Issues

**Objective**: Identify and document all data quality issues in the Netflix dataset including:
- Missing values
- Duplicates
- Malformed rows
- Inconsistent labels
- Outliers

**Author**: Netflix ML Pipeline  
**Date**: December 2025

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Load data
df = pd.read_csv('../data/raw/netflix1.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 1. Dataset Overview

In [ ]:
# Basic information
print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("MEMORY USAGE")
print("=" * 60)
memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Total memory usage: {memory_mb:.2f} MB")

## 2. Missing Values Analysis

In [ ]:
# Calculate missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values
}).sort_values('Missing Count', ascending=False)

print("Missing Values Summary:")
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualize missing values
fig = px.bar(
    missing_df[missing_df['Missing Count'] > 0],
    x='Column',
    y='Missing %',
    title='Missing Values by Column',
    labels={'Missing %': 'Percentage Missing'},
    color='Missing %',
    color_continuous_scale='Reds'
)
fig.update_layout(height=500)
fig.show()

## 3. Duplicate Analysis

In [ ]:
# Check duplicates
print("=" * 60)
print("DUPLICATE ANALYSIS")
print("=" * 60)

full_dupes = df.duplicated().sum()
print(f"Full duplicate rows: {full_dupes}")

if 'show_id' in df.columns:
    id_dupes = df.duplicated(subset=['show_id']).sum()
    print(f"Duplicate show_ids: {id_dupes}")
    
    if id_dupes > 0:
        print("\nDuplicate show_id examples:")
        dupe_ids = df[df.duplicated(subset=['show_id'], keep=False)]['show_id'].unique()[:5]
        for sid in dupe_ids:
            print(f"\nshow_id: {sid}")
            print(df[df['show_id'] == sid][['show_id', 'title', 'type']])

if 'title' in df.columns:
    title_dupes = df.duplicated(subset=['title']).sum()
    print(f"\nDuplicate titles: {title_dupes}")

## 4. Data Type Validation

In [ ]:
# Data types and unique values
dtype_summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Unique Values': [df[col].nunique() for col in df.columns],
    'Sample Value': [df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None for col in df.columns]
})

print("Data Type Summary:")
print(dtype_summary)

## 5. Categorical Consistency

In [ ]:
# Check 'type' column
if 'type' in df.columns:
    print("Type Distribution:")
    print(df['type'].value_counts())
    
    fig = px.pie(
        values=df['type'].value_counts().values,
        names=df['type'].value_counts().index,
        title='Content Type Distribution'
    )
    fig.show()

In [ ]:
# Check 'rating' column
if 'rating' in df.columns:
    print("Rating Distribution:")
    rating_counts = df['rating'].value_counts()
    print(rating_counts)
    
    fig = px.bar(
        x=rating_counts.index,
        y=rating_counts.values,
        title='Rating Distribution',
        labels={'x': 'Rating', 'y': 'Count'}
    )
    fig.update_layout(height=500)
    fig.show()

## 6. Date Format Validation

In [ ]:
# Check date_added
if 'date_added' in df.columns:
    print("Date Added Analysis:")
    print(f"Sample values: {df['date_added'].dropna().head(10).tolist()}")
    
    # Try parsing
    parsed_dates = pd.to_datetime(df['date_added'], errors='coerce')
    invalid_dates = parsed_dates.isnull().sum() - df['date_added'].isnull().sum()
    
    print(f"\nInvalid date formats: {invalid_dates}")
    print(f"Date range: {parsed_dates.min()} to {parsed_dates.max()}")
    
    # Plot timeline
    if invalid_dates == 0:
        date_counts = parsed_dates.dt.to_period('M').value_counts().sort_index()
        
        fig = px.line(
            x=date_counts.index.astype(str),
            y=date_counts.values,
            title='Content Additions Over Time',
            labels={'x': 'Month', 'y': 'Number of Titles Added'}
        )
        fig.update_layout(height=500)
        fig.show()

In [ ]:
# Check release_year
if 'release_year' in df.columns:
    print("Release Year Analysis:")
    print(f"Range: {df['release_year'].min()} to {df['release_year'].max()}")
    print(f"Mean: {df['release_year'].mean():.2f}")
    print(f"Median: {df['release_year'].median()}")
    
    # Check for outliers
    outliers = df[(df['release_year'] < 1900) | (df['release_year'] > 2025)]
    print(f"\nOutliers (< 1900 or > 2025): {len(outliers)}")
    
    if len(outliers) > 0:
        print(outliers[['title', 'release_year']])

## 7. Duration Format Validation

In [ ]:
# Check duration formats
if 'duration' in df.columns:
    print("Duration Format Analysis:")
    print(f"Sample values: {df['duration'].dropna().head(10).tolist()}")
    
    has_min = df['duration'].str.contains('min', na=False).sum()
    has_season = df['duration'].str.contains('Season', na=False).sum()
    total_non_null = df['duration'].notna().sum()
    malformed = total_non_null - has_min - has_season
    
    print(f"\nMinutes format: {has_min}")
    print(f"Seasons format: {has_season}")
    print(f"Malformed: {malformed}")
    
    if malformed > 0:
        print("\nMalformed duration examples:")
        malformed_mask = (~df['duration'].str.contains('min', na=False)) & \
                        (~df['duration'].str.contains('Season', na=False)) & \
                        (df['duration'].notna())
        print(df[malformed_mask][['title', 'type', 'duration']].head(10))

## 8. Outlier Detection

In [ ]:
# Release year outliers
if 'release_year' in df.columns:
    Q1 = df['release_year'].quantile(0.25)
    Q3 = df['release_year'].quantile(0.75)
    IQR = Q3 - Q1
    
    outlier_mask = (df['release_year'] < Q1 - 1.5 * IQR) | \
                   (df['release_year'] > Q3 + 1.5 * IQR)
    
    print(f"Release Year Outliers (IQR method):")
    print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"Outliers: {outlier_mask.sum()} ({outlier_mask.sum()/len(df)*100:.2f}%)")
    
    # Box plot
    fig = px.box(
        df,
        y='release_year',
        title='Release Year Distribution (Box Plot)'
    )
    fig.show()

## 9. Text Field Quality

In [ ]:
# Analyze text fields
text_columns = ['title', 'director', 'cast', 'country', 'listed_in']

text_quality = []
for col in text_columns:
    if col in df.columns:
        col_data = df[col].dropna()
        text_quality.append({
            'Column': col,
            'Avg Length': col_data.str.len().mean(),
            'Max Length': col_data.str.len().max(),
            'Min Length': col_data.str.len().min(),
            'Empty Strings': (col_data == '').sum()
        })

text_quality_df = pd.DataFrame(text_quality)
print("Text Field Quality:")
print(text_quality_df)

## 10. Generate Automated Report

In [ ]:
# Run automated quality check
from src.data.quality_check import DataQualityChecker

checker = DataQualityChecker(
    data_path='../data/raw/netflix1.csv',
    output_dir='../reports/data_quality'
)

report = checker.run_full_assessment()
checker.save_report()

print("\n✓ Automated quality report generated")

## Summary

### Key Findings:
1. **Missing Values**: Director, cast, country, and date_added have significant missing values
2. **Duplicates**: Check for duplicate show_ids and titles
3. **Date Formats**: date_added needs parsing and standardization
4. **Duration**: Two formats exist (minutes for movies, seasons for TV shows)
5. **Ratings**: Multiple rating systems (TV and movie ratings)

### Next Steps:
1. Implement deterministic cleaning pipeline
2. Handle missing values with appropriate strategies
3. Normalize categorical fields
4. Parse and standardize dates
5. Convert durations to numeric features